# EndoScan ER — Phase 2b operator runbook (FOUR STOPS — *not* Run-all)

**You are the operator. You never edit Python.** You only **paste values** into
the clearly-marked CONFIG cells in Stop 2 and Stop 3. Run the cells of one stop,
then **STOP**: copy the printed outputs into Claude Chat and wait for the next
instruction. Do **not** run the next stop until Claude Chat tells you to.

**Private repo, read-only access.** This notebook clones the repo **read-only** and
**never pushes to GitHub**. The only credential it uses is a **read-only,
single-repo** fine-grained PAT stored as the Colab Secret **`GH_PAT_RO`** — the
**Preflight** cell checks it before anything else. The token is never printed and is
never written to a URL, argv, or `.git/config`.

- **Preflight** — verify the read-only PAT, before any install/clone/download.
- **Stop 1** — clone (read-only) + **light** inspect of the metadata + CERAPP archives
  (NOT the 19.9 GB gctx); report the expected gctx shape from its filename.
- **Stop 2** — paste the column names Claude Chat gives you; **then** download the gctx
  (heavy, ~20 GB) and build the dataset + run the **quality gate only** (no training).
- **Stop 3** — paste the metric floors + approval Claude Chat gives you; train.
- **Stop 4** — only after Claude Chat approves: DVC-push the binaries to Google
  Drive and download a small **review bundle**. **No GitHub push happens here** —
  you hand the bundle to Claude Code, which opens the review PR.

EndoScan is a pre-screening / prioritization tool — **no regulatory-grade claims**.

## PREFLIGHT — verify the read-only PAT (run FIRST)
Before any install, clone, or download. Set the Colab Secret **`GH_PAT_RO`** to a
fine-grained PAT scoped to the single repo `Rirkella/endoscan-platform` with
**Contents: Read-only** and a short expiry. The token is checked here and never
printed.

In [ ]:
# PREFLIGHT — verify the read-only single-repo PAT BEFORE install/clone/download.
# Defines REPO + TOKEN used by the Stop 1 clone. Never prints the token or the raw
# response; only "repo access OK" or a clear failure reason.
import requests
from google.colab import userdata

REPO = 'Rirkella/endoscan-platform'
try:
    TOKEN = (userdata.get('GH_PAT_RO') or '').strip()
except Exception:
    TOKEN = ''
if not TOKEN:
    raise SystemExit(
        'Set Colab Secret GH_PAT_RO to a fine-grained PAT '
        '(single repo Rirkella/endoscan-platform, Contents: Read-only, short expiry).')

resp = requests.get(
    f'https://api.github.com/repos/{REPO}',
    headers={'Authorization': f'Bearer {TOKEN}', 'Accept': 'application/vnd.github+json'},
    timeout=30,
)
if resp.status_code == 200:
    print('repo access OK')
else:
    raise SystemExit(
        f'repo access FAILED (HTTP {resp.status_code}). Check that GH_PAT_RO is a '
        'fine-grained PAT for this single repo with Contents: Read-only and is not expired.')

## STOP 1 — Clone (read-only) + LIGHT inspect (no 20 GB gctx)
Run cells **1a → 1a-check → 1b → 1c** in order, then STOP. The clone is read-only and
token-safe; **1a-check** confirms `endoscan_core` imports in-kernel before any fetch.
**1c is light:** it fetches + inspects the metadata (`sig_info`, `gene_info`,
`pert_info`) and the CERAPP experimental archives, and reports the **expected** gctx
shape from its filename — it does **not** download the 19.9 GB gctx (that happens in
Stop 2, after the columns are confirmed). 1c **halts** if no readable CERAPP table is
found. Cell **1d** is a fallback to run *only* if 1c reports the CERAPP fetch failed.

In [ ]:
# 1a) Environment + token-safe READ-ONLY clone.
# Standard CPU runtime: install ONLY numpy-2-compatible packages and do NOT touch
# numpy. h5py/pandas/pyarrow are Colab's preinstalled numpy-2 builds — we don't
# reinstall them, so there is no ABI break and no runtime restart.
import os, stat, subprocess, sys
from pathlib import Path

!pip -q install 'scikit-learn>=1.8' pyarrow dvc requests

url = f'https://github.com/{REPO}.git'  # tokenless URL — the PAT is supplied via GIT_ASKPASS
# Transient GIT_ASKPASS helper: reads the token from the env (NOT argv/URL/config) and
# emits a throwaway username + the token. The token never touches the URL, argv,
# .git/config, the shell, or any printed output.
askpass = Path('/tmp/gh_askpass.sh')
askpass.write_text('#!/usr/bin/env bash\ncase "$1" in\n  *Username*) echo "x-access-token";;\n  *) echo "$GH_PAT_RO";;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IEXEC)
clone_env = {**os.environ, 'GH_PAT_RO': TOKEN, 'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0'}
subprocess.run(['git', 'clone', '--depth', '1', url], check=True, env=clone_env)
# Reset origin to the tokenless URL (defensive) and remove the helper immediately.
subprocess.run(['git', 'remote', 'set-url', 'origin', url], cwd='endoscan-platform', check=True)
askpass.unlink(missing_ok=True)

%cd endoscan-platform
# Editable install with the SAME interpreter as the kernel (sys.executable), NOT bare
# !pip. pip writes a .pth/finder that is only processed at interpreter startup, so the
# running kernel can't import the editable package until we refresh its import state.
!{sys.executable} -m pip install -q -e packages/endoscan_core
import site, importlib
site.main(); importlib.invalidate_caches()  # process the new .pth in THIS kernel
sys.path.insert(0, os.path.abspath('packages/endoscan_core'))  # deterministic fallback
sys.path.insert(0, 'pipelines/endpoints/ER/staging')

# Verify a clean numpy-2 stack (no ABI break) before any data work.
import numpy, pandas, h5py, pyarrow
print('imports OK — numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| h5py', h5py.__version__, '| pyarrow', pyarrow.__version__)
print('Cloned read-only; origin =', url)

In [ ]:
# 1a-check) IMPORT PREFLIGHT — confirm endoscan_core imports IN-KERNEL before any fetch.
# This runs in the kernel itself (NOT a `!python -c` subprocess), so it catches the
# editable-install refresh problem a subprocess check would falsely pass.
import os
from pathlib import Path

print('cwd:', os.getcwd())
if not Path('packages/endoscan_core').is_dir():
    raise SystemExit(
        'packages/endoscan_core not found in the current directory — run cell 1a '
        '(clone + install) first; the kernel must be in the cloned repo root.')
try:
    import endoscan_core  # noqa: F401 — in-kernel import (NOT a !python -c subprocess)
    from endoscan_core.datasets import load_sources  # noqa: F401
except ImportError as exc:
    raise SystemExit(
        f'endoscan_core import FAILED in-kernel ({exc}). The install cell did not '
        'complete — re-run cell 1a (it installs -e and refreshes the kernel import state).')
print('endoscan_core import OK')

In [ ]:
# 1b) Mount Google Drive and configure the DVC LOCAL remote (no credentials in repo;
#     the path lives in git-ignored .dvc/config.local).
from google.colab import drive; drive.mount('/content/drive')
!dvc remote add --local er_gdrive /content/drive/MyDrive/endoscan-dvc || true
!dvc remote default --local er_gdrive
print('DVC local remote -> /content/drive/MyDrive/endoscan-dvc')

In [ ]:
# 1c) LIGHT inspect — metadata + CERAPP only. The 19.9 GB gctx is NOT downloaded here;
# it is fetched in Stop 2 after columns are confirmed (only then do we know which
# sig_ids to slice). Nothing is staged or trained.
from pathlib import Path
import gzip, pandas as pd
import fetch
from endoscan_core.datasets import load_sources

loc = {l.name: l.url for s in load_sources().sources for l in s.locators}
raw = Path('data/raw'); raw.mkdir(parents=True, exist_ok=True)

# (1-3) Three LINCS metadata files. pert_info is included because it carries the
# perturbagen -> InChIKey link used to match LINCS signatures to CERAPP compounds.
small = {
    'sig_info':  loc['gse92742_sig_info'],
    'gene_info': loc['gse92742_gene_info'],
    'pert_info': loc['gse92742_pert_info'],
}
for name, url in small.items():
    fetch.fetch_url(url, raw / f'{name}.txt.gz')
    head = pd.read_csv(raw / f'{name}.txt.gz', sep='\t', nrows=5)
    n = sum(1 for _ in gzip.open(raw / f'{name}.txt.gz', 'rt')) - 1
    print(f'\n=== {name}: {n} rows ===\ncolumns: {list(head.columns)}'); display(head.head())

# Landmark count from gene_info (rows with pr_is_lm set) — the ~978 figure.
gene_info = pd.read_csv(raw / 'gene_info.txt.gz', sep='\t')
if 'pr_is_lm' in gene_info.columns:
    landmark_count = int(gene_info['pr_is_lm'].astype(str).isin({'1', 'True', 'true'}).sum())
else:
    landmark_count = 'unknown (confirm GENE_LM_COL with Claude Chat at Stop 2)'
print('\nlandmark genes (pr_is_lm set):', landmark_count)

# (4) CERAPP EXPERIMENTAL labels — resolve (GAFTP mirror, then figshare files[] DATA
# file — never the article HTML page), VALIDATE each artifact (reject HTML / require a
# real ZIP; stale invalid caches re-fetched), EXTRACT, and list EVERY tabular file
# (delimiter sniffed). Do NOT auto-pick: Claude Chat selects the experimental table.
cerapp_article_ids = [loc[k].rstrip('/').split('/')[-1]
                      for k in loc if 'cerapp' in k and 'figshare.com' in loc[k]]
gaftp_url = next((loc[k] for k in loc if 'gaftp' in k), None)
print('\nCERAPP figshare article ids:', cerapp_article_ids, '| gaftp mirror:', bool(gaftp_url))
cerapp_src = raw / 'cerapp_src'
cerapp_manifest = []
try:
    cerapp_files = fetch.fetch_cerapp_experimental(cerapp_src, cerapp_article_ids,
                                                   gaftp_url=gaftp_url)
    print('Downloaded + validated CERAPP artifacts:', [p.name for p in cerapp_files])
    print('\n=== CERAPP tabular files (relative to data/raw/cerapp_src) ===')
    cerapp_manifest = fetch.inspect_cerapp_archives(cerapp_files, cerapp_src)
except Exception as exc:
    # The error distinguishes "HTML / not a valid ZIP" from "could not download".
    print('CERAPP RESOLVE/VALIDATE FAILED:', exc)

# GUARD — no readable experimental table => HALT. Nothing downstream may download the
# gctx until CERAPP inspection succeeds (the heavy gctx fetch lives in Stop 2).
if not cerapp_manifest:
    raise SystemExit(
        'STOP 1c HALTED: no readable CERAPP experimental table after extraction. '
        'Fix the source or run the manual-upload fallback cell (1d), then re-run 1c. '
        'The 19.9 GB gctx is NOT downloaded until this succeeds.')

# Expected gctx shape from the filename ONLY (no download / no decompress here).
gctx_name = Path(loc['gse92742_level5_modz_gctx']).name
dims = fetch.parse_gctx_dims(gctx_name)  # (n_signatures, n_genes)
gctx_expected = (f'{dims[1]} genes x {dims[0]} signatures' if dims else 'unknown (filename unparsed)')

print('\n================ STOP 1 SUMMARY ================')
print('metadata: sig_info / gene_info / pert_info fetched + inspected above')
print('landmark genes (pr_is_lm):', landmark_count)
print('CERAPP tables found:', len(cerapp_manifest),
      '->', [m['rel'] for m in cerapp_manifest])
print('gctx (EXPECTED from filename, NOT downloaded):', gctx_expected,
      f'| landmark={landmark_count}')
print('gctx file:', gctx_name)
print('================================================')

In [ ]:
# 1d) FALLBACK — ONLY run this if 1c printed "AUTOMATED CERAPP FETCH FAILED".
#     The automated figshare/gaftp resolution in 1c is the primary path; this manual
#     upload is the documented fallback for when neither source resolves. Upload the
#     CERAPP EXPERIMENTAL table (.csv/.xlsx) when prompted, then re-run cell 1c.
from pathlib import Path
import pandas as pd
from google.colab import files
raw = Path('data/raw'); raw.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
src = Path(next(iter(uploaded)))
df = pd.read_excel(src) if src.suffix.lower() in {'.xlsx', '.xls'} else pd.read_csv(src)
df.to_csv(raw / 'cerapp_experimental.csv', index=False)
print('Wrote', raw / 'cerapp_experimental.csv', '— now re-run cell 1c to inspect it.')

### ⛔ STOP — send Claude Chat:
the printed **headers + row counts** for `sig_info`, `gene_info`, `pert_info`, the
**landmark count** (`pr_is_lm`), the **CERAPP table manifest** (every tabular file
found, with delimiter + experimental/consensus note), and the **EXPECTED gctx shape**
(parsed from the filename — the 20 GB gctx is downloaded in **Stop 2**, not now).
**Do not proceed** until Claude Chat replies with the chosen **CERAPP experimental
table** and the confirmed column names.

## STOP 2 — Paste reviewed columns, stage, gate ONLY
Paste the values Claude Chat gave you into the CONFIG cell, run both cells, then
STOP. **Training cannot happen in this stop** (approval is hard-wired to False).

In [ ]:
# 2a) CONFIG — paste ONLY these values (column names Claude Chat confirmed). No logic.
SIG_ID_COL     = ''   # sig_info: signature id
SIG_PERT_COL   = ''   # sig_info: perturbagen/compound id
SIG_CELL_COL   = ''   # sig_info: cell line
SIG_DOSE_COL   = ''   # sig_info: dose
SIG_TIME_COL   = ''   # sig_info: time
GENE_LM_COL    = ''   # gene_info: landmark flag column
GENE_ID_COL    = ''   # gene_info: gene id
GENE_SYM_COL   = ''   # gene_info: gene symbol
PERT_ID_COL    = ''   # pert_info: perturbagen id
PERT_INCHI_COL = ''   # pert_info: InChIKey
CERAPP_TABLE        = ''  # cerapp: chosen EXPERIMENTAL table — rel path from the Stop 1 manifest
                          #         (under data/raw/cerapp_src). Leave '' to use a 1d manual upload.
# CERAPP is assay-level (many rows per CASRN) -> collapsed to ONE ER label per compound.
CERAPP_LABEL_MODE      = 'binding'           # 'binding' (ER binding-class only) | 'any' (all assay classes)
CERAPP_CASRN_COL       = 'CASRN'             # cerapp: CASRN column
CERAPP_ASSAY_CLASS_COL = 'ASSAY_CLASS_NAME'  # cerapp: assay-class column (the binding filter)
CERAPP_ACTIVE_COL      = 'All_active'        # cerapp: 0/1 active-call column
CERAPP_INCHI_COL       = 'InChI_Code'        # cerapp: InChI string (provenance only)

In [ ]:
# 2b) Build staged files (tested helpers) and run the GATE ONLY (no training).
import importlib.util, pandas as pd
from pathlib import Path
import cerapp, pubchem, stage_er, fetch
from endoscan_core.datasets import load_sources

raw = Path('data/raw'); staged = Path('data/staged/er'); staged.mkdir(parents=True, exist_ok=True)
loc = {l.name: l.url for s in load_sources().sources for l in s.locators}

# Labels: CERAPP EXPERIMENTAL calls only (never consensus predictions). Read the chosen
# table (xlsx via openpyxl, or sniffed csv/txt) — or the 1d manual upload if '' .
cerapp_table = (raw / 'cerapp_src' / CERAPP_TABLE) if CERAPP_TABLE else (raw / 'cerapp_experimental.csv')
_read = fetch.read_tabular(cerapp_table)
assert _read is not None, f'CERAPP table is not readable: {cerapp_table} (check CERAPP_TABLE)'
cerapp_rows = _read[0].to_dict('records')
# The evaluation set is ASSAY-LEVEL -> collapse to ONE ER label per compound. Compounds
# whose assays DISAGREE are EXCLUDED (never majority-voted) and reported, not staged.
labels, conflicts = cerapp.assemble_evaluation_labels(
    cerapp_rows, casrn_col=CERAPP_CASRN_COL, assay_class_col=CERAPP_ASSAY_CLASS_COL,
    active_col=CERAPP_ACTIVE_COL, inchi_col=CERAPP_INCHI_COL, label_mode=CERAPP_LABEL_MODE)
pd.DataFrame(labels).to_csv(staged / 'cerapp.csv', index=False)
print(f'CERAPP label_mode={CERAPP_LABEL_MODE}: {len(labels)} compound labels '
      f'| {len(conflicts)} conflicts EXCLUDED (disagreeing assays)')

# CASRN -> InChIKey for the labelled compounds (PubChem REST), normalized + saved.
casrns = sorted({r['casrn'] for r in labels})
mapping_rows = pubchem.normalize_mapping(fetch.pubchem_mapping_for_casrns(casrns))
pd.DataFrame(mapping_rows).to_csv(staged / 'pubchem.csv', index=False)
casrn_to_inchi = {m['input_id']: m['inchikey'] for m in mapping_rows}
labelled_inchikeys = {casrn_to_inchi[r['casrn']] for r in labels if r['casrn'] in casrn_to_inchi}

# Build the needed sig_id list FIRST — sig_info filtered to CERAPP-labelled compounds in
# MCF7/A549 (perturbagen -> InChIKey from pert_info). Only now do we know what to slice.
pert_info = pd.read_csv(raw / 'pert_info.txt.gz', sep='\t')
pert_to_inchi = dict(zip(pert_info[PERT_ID_COL].astype(str), pert_info[PERT_INCHI_COL].astype(str)))
sig_info = pd.read_csv(raw / 'sig_info.txt.gz', sep='\t')
sig_meta = stage_er.assemble_sig_meta(sig_info, pert_to_inchi, labelled_inchikeys,
    sig_id_col=SIG_ID_COL, pert_id_col=SIG_PERT_COL, cell_id_col=SIG_CELL_COL,
    dose_col=SIG_DOSE_COL, time_col=SIG_TIME_COL)
gene_info = pd.read_csv(raw / 'gene_info.txt.gz', sep='\t')
gene_ids, gene_syms = stage_er.select_landmark_genes(gene_info, landmark_flag_col=GENE_LM_COL,
    gene_id_col=GENE_ID_COL, gene_symbol_col=GENE_SYM_COL)

# HEAVY STEP — only now (sig_ids known) download + decompress the Level-5 gctx.
# ~20 GB download, ~40 GB decompressed, ~20-40 min. Resumes from a valid cached .gctx;
# a truncated/HTML/invalid cache is discarded and re-fetched.
gctx_path = raw / 'level5_modz.gctx'
print(f'HEAVY: fetching the Level-5 gctx to slice {sig_meta["sig_id"].nunique()} sig_ids x '
      f'{len(gene_ids)} landmark genes (~20 GB download, ~40 GB decompressed, ~20-40 min)...')
fetch.download_gctx(loc['gse92742_level5_modz_gctx'], gctx_path, gz_path=raw / 'level5_modz.gctx.gz')

# Slice the gctx (partial HDF5 read of only those sig_ids x 978 landmark rows) -> fuse.
stage_er.build_lincs_parquet(sig_meta, gctx_path, gene_ids, gene_syms, staged / 'lincs.parquet')

# Staged-data sanity check on the transcriptomics matrix the model will train on.
lincs = pd.read_parquet(staged / 'lincs.parquet')
feat_cols = [c for c in lincs.columns if c != 'compound_key']
print(f'\n=== lincs.parquet: {lincs.shape[0]} compounds x {len(feat_cols)} landmark genes ===')
assert feat_cols == list(gene_syms), 'feature columns must be the landmark gene symbols, in order'
assert not lincs[feat_cols].isna().any().any(), 'no NaNs allowed in the landmark matrix'
print('columns are landmark gene symbols:', feat_cols[:5], '...',
      '| NaNs:', int(lincs[feat_cols].isna().sum().sum()))

# GATE ONLY — approval hard-wired False here, so run_pipeline cannot train.
spec = importlib.util.spec_from_file_location('er_run', 'pipelines/endpoints/ER/run.py')
er_run = importlib.util.module_from_spec(spec); sys.modules['er_run'] = er_run; spec.loader.exec_module(er_run)
cfg = er_run.PipelineConfig.model_validate({
    'endpoint_id': 'ER', 'biological_target': 'Estrogen Receptor', 'version': '0.1.0',
    'data': {'target': 'ER', 'adapter': 'staged', 'staged_dir': 'data/staged/er', 'n_groups': 10, 'seed': 0},
    'gate': {'thresholds_path': 'registry/data/quality_gates.yaml'},
    'approval': {'approved': False},  # HARD BOUNDARY for Stop 2
    'evaluation': {'mode': 'nested', 'outer_splits': 5, 'inner_splits': 3},
})
res = er_run.run_pipeline(cfg, allow_list=load_sources(), data_dir=staged,
    thresholds=er_run.load_thresholds(Path('registry/data/quality_gates.yaml')), output_root=Path('.'))
assert res.trained is False  # structurally guaranteed in Stop 2
pos = sum(1 for r in labels if r['consensus_call'] == 'active')
print('gate:', res.gate_summary, '| overlap compounds:', res.n_overlap,
      '| labelled prevalence:', round(pos / max(len(labels), 1), 3))
print('\n=== dataset_card.md ===\n' + Path(res.dataset_card_path).read_text())

### ⛔ STOP — send Claude Chat:
the printed **`dataset_card.md`**, the **gate summary**, the **prevalence**, and the
**compound counts**. Training is intentionally unreachable here. Wait for Claude
Chat to provide the metric floors before Stop 3.

## STOP 3 — Paste floors + approval, train
Paste the floors Claude Chat gave you, run both cells, then STOP.

In [ ]:
# 3a) CONFIG — paste ONLY these values (validated_mvp floors Claude Chat provided).
FLOOR_AUROC             = 0.0   # <- paste
FLOOR_AUPRC             = 0.0   # <- paste (sized to prevalence)
FLOOR_BALANCED_ACCURACY = 0.0   # <- paste
CEIL_BRIER              = 1.0   # <- paste
APPROVED                = True  # Claude Chat authorizes training for this stop

In [ ]:
# 3b) Train: re-gates, trains (nested honest estimate), writes artifacts + proposed entry.
from pathlib import Path
from endoscan_core.datasets import load_sources
cfg = er_run.PipelineConfig.model_validate({
    'endpoint_id': 'ER', 'biological_target': 'Estrogen Receptor', 'version': '0.1.0',
    'data': {'target': 'ER', 'adapter': 'staged', 'staged_dir': 'data/staged/er', 'n_groups': 10, 'seed': 0},
    'gate': {'thresholds_path': 'registry/data/quality_gates.yaml'},
    'approval': {'approved': APPROVED, 'approved_by': 'operator+claude-chat'},
    'evaluation': {'mode': 'nested', 'outer_splits': 5, 'inner_splits': 3},
    'training': {'validated_mvp_floors': {'auroc': FLOOR_AUROC, 'auprc': FLOOR_AUPRC,
                                          'balanced_accuracy': FLOOR_BALANCED_ACCURACY},
                 'validated_mvp_ceilings': {'brier_score': CEIL_BRIER}},
})
res = er_run.run_pipeline(cfg, allow_list=load_sources(), data_dir=Path('data/staged/er'),
    thresholds=er_run.load_thresholds(Path('registry/data/quality_gates.yaml')), output_root=Path('.'))
print('trained:', res.trained, '| status:', res.status, '| model:', res.selected_model)
print('\n=== metrics.json ===\n' + Path('models/ER/metrics.json').read_text())
print('\n=== model_selection.md ===\n' + Path('models/ER/model_selection.md').read_text())

### ⛔ STOP — send Claude Chat:
`models/ER/metrics.json`, `model_selection.json`, `model_selection.md`,
`model_card.md`, `dataset_card.md`, `feature_schema.json`, and the proposed
`registry/models/endpoints.json` entry. **Do NOT push or commit yet.**

## STOP 4 — Only after Claude Chat approval: DVC-push binaries + download the bundle
This stop **never pushes to GitHub** and **never runs git**. It DVC-pushes the
binaries to the Google Drive remote and downloads a small review bundle (text
artifacts + `.dvc` pointers + the proposed `endpoints.json`). You hand that bundle
to **Claude Code**, which opens the `er-real-endpoint-phase2b` PR (text + pointers
only); Claude Chat reviews the diff and you merge. **No token is needed for this
step.**

In [ ]:
# 4) DVC-push the BINARIES to the Drive remote (NOT GitHub), then write + download a
#    small review bundle. This cell NEVER runs git, NEVER pushes to GitHub, NEVER opens
#    a PR. The only credential ever used by this notebook was the read-only PAT in
#    Preflight; this step needs no token at all.
import zipfile
from pathlib import Path
import bundle
from google.colab import files

# (a) DVC-push ONLY the large binaries to the Google Drive local remote.
!dvc add models/ER/model.pkl data/staged/er/lincs.parquet && dvc push

# (b) Assemble the review bundle on Drive: text artifacts + small CSVs + the two .dvc
#     pointer files + the proposed endpoints.json (written into the clone by Stop 3's
#     register_endpoint). Binaries stay in DVC/Drive — they are NOT in the bundle.
REL = [
    'models/ER/metrics.json', 'models/ER/feature_schema.json',
    'models/ER/model_card.md', 'models/ER/dataset_card.md',
    'models/ER/model_selection.json', 'models/ER/model_selection.md',
    'data/staged/er/cerapp.csv', 'data/staged/er/pubchem.csv',
    'models/ER/model.pkl.dvc', 'data/staged/er/lincs.parquet.dvc',
    'registry/models/endpoints.json',  # proposed entry (whole file; Claude Code diffs it)
]
bundle_dir = Path('/content/drive/MyDrive/endoscan-dvc/er_phase2b_bundle')
copied, missing = bundle.assemble_review_bundle(Path('.'), bundle_dir, REL)
print('bundled:', copied)
if missing:
    print('MISSING (skipped):', missing)

# (c) Zip the bundle and trigger a one-click download.
zip_path = '/content/er_phase2b_bundle.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in bundle_dir.rglob('*'):
        if p.is_file():
            z.write(p, p.relative_to(bundle_dir))
print('\nBundle ->', bundle_dir, '\nZipped ->', zip_path)
files.download(zip_path)

# (d) REFERENCE ONLY — do NOT run these here. Hand the bundle to Claude Code, which
#     runs the equivalent locally to open the er-real-endpoint-phase2b PR.
print('\n# Reference only (Claude Code runs this locally from the bundle; NOT in Colab):')
print('git checkout -b er-real-endpoint-phase2b')
print('git add models/ER/model.pkl.dvc data/staged/er/lincs.parquet.dvc \\')
print('        models/ER/metrics.json models/ER/feature_schema.json \\')
print('        models/ER/model_card.md models/ER/dataset_card.md \\')
print('        models/ER/model_selection.json models/ER/model_selection.md \\')
print('        data/staged/er/cerapp.csv data/staged/er/pubchem.csv \\')
print('        registry/models/endpoints.json')
print('git commit -m "ER real endpoint (Phase 2b): registered <status> with DVC pointers"')